# 07 Extract epoch-wise label time courses

Apply inverse operators to cleaned epochs and save compact label-wise source time courses per epoch.

This is intended as a source-space derivative for downstream connectivity and decoding. It does **not** save full vertex-wise source epochs.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from meeg_pipeline.config import load_config
from meeg_pipeline.workflow import iter_recordings, recordings_to_dataframe
from meeg_pipeline.source_modeling import (
    label_time_courses_epochs_config_to_dataframe,
    epoch_label_time_course_input_overview_to_dataframe,
    extract_epoch_label_time_courses_for_recordings,
    epoch_label_time_course_results_to_dataframe,
    epoch_label_time_course_qc_to_dataframe,
)


In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    """Find project root by searching upward for configs/local.yaml."""
    start = Path.cwd() if start is None else Path(start).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "local.yaml").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find project root by searching for configs/local.yaml "
        f"above {start}"
    )


PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "configs" / "local.yaml"
config = load_config(CONFIG_PATH)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CONFIG_PATH:", CONFIG_PATH)


## Run settings

Methodological settings are read from `configs/local.yaml` under `source.label_time_courses_epochs`. The flags here only control notebook execution.


In [ ]:
SUBJECTS = "all"
SESSIONS = "all"
TASKS = "all"
RUNS = "all"

OVERWRITE_STEPS = []

RUN_EXTRACTION = True
RUN_QC = True
MAX_QC_ROWS = None

# Useful for a quick smoke test before running the full batch.
MAX_RECORDINGS_FOR_TEST = None


In [ ]:
display(label_time_courses_epochs_config_to_dataframe(config))


## Select recordings

In [ ]:
selected_recordings = list(
    iter_recordings(
        config,
        subjects=SUBJECTS,
        sessions=SESSIONS,
        tasks=TASKS,
        runs=RUNS,
    )
)

if MAX_RECORDINGS_FOR_TEST is not None:
    selected_recordings = selected_recordings[:MAX_RECORDINGS_FOR_TEST]

recordings_df = recordings_to_dataframe(selected_recordings)
display(recordings_df)


## Input overview

In [ ]:
on_existing = "overwrite" if "epoch_label_time_courses" in OVERWRITE_STEPS else "skip"

epoch_ltc_overview = epoch_label_time_course_input_overview_to_dataframe(
    config,
    selected_recordings,
    on_existing=on_existing,
)

display(epoch_ltc_overview)

if not epoch_ltc_overview.empty:
    display(
        epoch_ltc_overview
        .groupby(["task", "status"], dropna=False)
        .size()
        .reset_index(name="n")
        .sort_values(["task", "status"])
    )


## Extract epoch-wise label time courses

In [ ]:
if RUN_EXTRACTION:
    epoch_ltc_results = extract_epoch_label_time_courses_for_recordings(
        config,
        selected_recordings,
        on_existing=on_existing,
        verbose=True,
    )
    epoch_ltc_results_df = epoch_label_time_course_results_to_dataframe(epoch_ltc_results)
else:
    print("Skipped epoch-wise label-time-course extraction.")
    epoch_ltc_results_df = pd.DataFrame()

display(epoch_ltc_results_df)

if not epoch_ltc_results_df.empty:
    display(
        epoch_ltc_results_df
        .groupby(["task", "status"], dropna=False)
        .size()
        .reset_index(name="n")
        .sort_values(["task", "status"])
    )


## QC

In [ ]:
if RUN_QC:
    epoch_ltc_qc = epoch_label_time_course_qc_to_dataframe(
        epoch_ltc_results_df,
        max_rows=MAX_QC_ROWS,
    )
else:
    print("Skipped epoch-wise label-time-course QC.")
    epoch_ltc_qc = pd.DataFrame()

display(epoch_ltc_qc)


## Output reminder

The main output array has shape:

`n_epochs × n_labels × n_times`

with sidecars for label names, time points, and epoch/event metadata.
